<a href="https://www.kaggle.com/code/masum1834e/mobilenet?scriptVersionId=323709792" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# MobileNet

MobileNet is a family of lightweight deep learning models designed for **mobile devices**, **embedded systems**, and environments with limited computing power. Traditional CNNs like VGGNet or ResNet achieve high accuracy but require heavy computation and memory. MobileNet solves this by making CNNs **smaller**, **faster**, and **more efficient**. It was developed by researchers at Google.

## Why MobileNet Was Created

Deep neural networks usually need:

* Large GPUs
* High memory
* Long inference time

But smartphones, IoT devices, Raspberry Pi, drones, and edge devices cannot handle huge models efficiently.

MobileNet focuses on:

* Low latency
* Small model size
* Low power consumption
* Real-time inference

Applications include:

* Face detection
* Object recognition
* Mobile camera AI
* Gesture recognition
* Autonomous robots
* Medical devices

## Main Idea Behind MobileNet

Traditional CNNs (like VGGNet or ResNet) use **standard convolutions**, which are computationally expensive. MobileNet replaces these with a much cheaper operation called: **Depthwise Separable Convolution**

It splits a standard convolution into two steps:

1. Depthwise Convolution
    * Applies **one filter per input channel**
    * Captures spatial features (height & width)
2. Pointwise Convolution (1×1 convolution)
    * Combines outputs across channels
    * Builds feature relationships

**Why This Matters**

Compared to standard convolution:

* **~8–9× fewer computations**
* **Much smaller model size**
* Slight trade-off in accuracy

**Mathematical Comparison**

Suppose:

* Input feature map size = `D × D × M`
* Output channels = `N`
* Kernel size = `K × K`

Standard Convolution Cost

$$
K^2 \times M \times N \times D^2
$$

Depthwise Separable Convolution Cost

Depthwise:

$$
K^2 \times M \times D^2
$$

Pointwise:

$$
M \times N \times D^2
$$

Total:

$$
K^2MD^2 + MND^2
$$

Computational Savings

For `3×3` kernels:

MobileNet reduces computation by approximately:

$$
8 \text{ to } 9 \text{ } times
$$

compared to standard convolution.

### Standard Convolution (Traditional CNN)

Suppose:

* Input image: `224 × 224 × 3`
* Kernel size: `3 × 3`
* Number of filters: `64`

A normal convolution performs:

1. Spatial filtering
2. Channel combination

all at once. This is computationally expensive.

### Depthwise Separable Convolution

This replaces standard convolution with a much cheaper operation. Apply one filter per input channel.

If input has 3 channels:

* One filter for Red
* One for Green
* One for Blue

This extracts spatial features separately.

### Pointwise Convolution

* Then use `1 × 1` convolution to combine channels.
* This mixes information across channels.

### Visual Intuition

Traditional convolution:

```text
Input → Big convolution → Output
```

MobileNet:

```text
Input
   ↓
Depthwise convolution
   ↓
Pointwise convolution
   ↓
Output
```

This dramatically reduces computation.


### Example Workthrough

Assume a single 2D feature map with **2 channels**:

**Input tensor (3×3×2)**

Channel 1 (C1):

```
1  2  3
4  5  6
7  8  9
```

Channel 2 (C2):

```
9  8  7
6  5  4
3  2  1
```

We will use:

* Kernel size = **2 × 2**
* Stride = **1**
* No padding

So output size = **2 × 2**

#### STANDARD CONVOLUTION

Each filter sees:

> ALL channels together

So kernel shape = **2 × 2 × 2**

We use **1 filter only**.

**Filter values:**

Channel 1 kernel:

```
1  0
0  1
```

Channel 2 kernel:

```
1  1
1  0
```

**Output calculation**

C1:

```
1 2
4 5
```

C2:

```
9 8
6 5
```

Multiply & sum:

C1:

$$
1*1 + 2*0 + 4*0 + 5*1 = 1 + 0 + 0 + 5 = 6
$$
C2:

$$
9*1 + 8*1 + 6*1 + 5*0 = 9 + 8 + 6 + 0 = 23
$$

Final output:

$$
6 + 23 = 29
$$

Final standard convolution output:

```
29  28
23  20
```

#### DEPTHWISE CONVOLUTION

Each channel has its **own filter**

So:

* C1 has one filter
* C2 has one filter

No mixing between channels.


**Filters (same 2×2 each channel)**

```
1 0
0 1
```

**Output Calculation**

Input C1:

```
1 2
4 5
```

Compute:

```
1*1 + 2*0 + 4*0 + 5*1 = 1 + 5 = 6
```

C1 output:

```
6   8
12  14
```

C2 output:

```
14 12
8  6
```

No channel mixing happens.

#### POINTWISE CONVOLUTION (1×1)

We use depthwise output:

C1:

```
6   8
12  14
```

C2:

```
14 12
8  6
```

At each pixel, we combine channels. We use **two filters → output 2 channels**

Filter 1:

```
C1: 0.5
C2: 0.5
```

Filter 2:

```
C1: 1
C2: -1
```

**(1,1) position**

Input vector:

$$
C1 = 6,  C2 = 14
$$

Output channel 1:

```
6*0.5 + 14*0.5 = 3 + 7 = 10
```

Output channel 2:

```
6*1 + 14*(-1) = 6 - 14 = -8
```

**Final pointwise output:**

Channel 1:
```
10  10
10  10
```

Channel 2:
```
-8  -4
4    8
```

#### FINAL COMPARISON

| Type           | What it does                      |
| -------------- | --------------------------------- |
| Standard conv  | Mixes space + channels together   |
| Depthwise conv | Processes each channel separately |
| Pointwise conv | Mixes channels at each pixel      |


## MobileNet Architecture

### MobileNetV1

Main components:

* Depthwise separable convolutions
* Batch normalization
* ReLU activation

Structure:

```text
Conv → Depthwise Conv → Pointwise Conv
      ↓
      Repeat many times
      ↓
Global Average Pooling
      ↓
Softmax classifier
```

In [1]:
from tensorflow.keras.layers import DepthwiseConv2D, Conv2D, BatchNormalization, ReLU

def depthwise_separable_block(x, filters, stride):
    
    # Depthwise Convolution
    x = DepthwiseConv2D(
        kernel_size=3,
        strides=stride,
        padding='same',
        use_bias=False
    )(x)
    x = BatchNormalization()(x)
    x = ReLU(max_value=6.0)(x)

    # Pointwise Convolution
    x = Conv2D(
        filters,
        kernel_size=1,
        padding='same',
        use_bias=False
    )(x)
    x = BatchNormalization()(x)
    x = ReLU(max_value=6.0)(x)

    return x

2026-06-01 14:59:38.534652: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780325978.844745      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780325978.931484      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780325979.654841      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780325979.654900      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780325979.654903      16 computation_placer.cc:177] computation placer alr

In [2]:
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, ReLU
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

def MobileNet(input_shape=(224, 224, 3), num_classes=1000):

    inputs = Input(shape=input_shape)

    x = Conv2D(32, 3, strides=2, padding='same', use_bias=False)(inputs)
    x = BatchNormalization()(x)
    x = ReLU(max_value=6.0)(x)

    x = depthwise_separable_block(x, 64, 1)
    x = depthwise_separable_block(x, 128, 2)
    x = depthwise_separable_block(x, 128, 1)
    x = depthwise_separable_block(x, 256, 2)
    x = depthwise_separable_block(x, 256, 1)
    x = depthwise_separable_block(x, 512, 2)

    for _ in range(5):
        x = depthwise_separable_block(x, 512, 1)

    x = depthwise_separable_block(x, 1024, 2)
    x = depthwise_separable_block(x, 1024, 1)

    x = GlobalAveragePooling2D()(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    return Model(inputs, outputs)

In [3]:
model = MobileNet()
model.summary()

2026-06-01 15:00:09.683325: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 112, 112, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 112, 112, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (None, 112, 112, 32)   │           288 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_1              │ (None, 56, 56, 64)     │           576 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 56, 56, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_2              │ (None, 56, 56, 128)    │         1,152 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_5 (ReLU)                  │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 56, 56, 128)    │        16,384 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 4,253,864 (16.23 MB)

 Trainable params: 4,231,976 (16.14 MB)

 Non-trainable params: 21,888 (85.50 KB)

In [4]:
# Pre-trained MobileNet
import tensorflow as tf
from tensorflow.keras.applications import MobileNet

model = MobileNet(weights='imagenet')

# This loads MobileNet trained on ImageNet (a large dataset of labeled images).

17225924/17225924 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


**Parameters and possible values**

| Parameter               | Description                            | Possible values                         |
| ----------------------- | -------------------------------------- | --------------------------------------- |
| `input_shape`           | Shape of input images                  | Tuple like `(224,224,3)`, `(128,128,3)` |
| `alpha`                 | Width multiplier (controls model size) | `0.25`, `0.5`, `0.75`, `1.0`            |
| `depth_multiplier`      | Depthwise convolution multiplier       | Integer, usually `1`                    |
| `dropout`               | Dropout rate before classifier         | Float like `0.001`, `0.2`, `0.5`        |
| `include_top`           | Include final classification layer     | `True` or `False`                       |
| `weights`               | Pretrained weights                     | `'imagenet'`, `None`                    |
| `input_tensor`          | Existing Keras tensor as input         | Tensor object or `None`                 |
| `pooling`               | Pooling mode when `include_top=False`  | `None`, `'avg'`, `'max'`                |
| `classes`               | Number of output classes               | Any positive integer                    |
| `classifier_activation` | Activation for final layer             | `'softmax'`, `None`                     |

**ImageNet**

ImageNet is a massive image dataset used to train and evaluate computer vision models.

It contains:

* Millions of labeled images
* Thousands of object categories
* Human-annotated labels like:
    * cats
    * dogs
    * cars
    * bicycles
    * fruits
    * people

Why it is important

ImageNet became famous because it powered breakthroughs in deep learning and image classification. Many pretrained models in TensorFlow and Keras are trained on ImageNet, including:

* MobileNet
* ResNet
* VGG16
* EfficientNet
* InceptionV3

In [5]:
# Feature Extraction
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

# Load pretrained MobileNet without classifier head
base_model = MobileNet(
    include_top=False,
    input_shape=(224, 224, 3),
    weights="imagenet"
)

# Add custom classification head
x = GlobalAveragePooling2D()(base_model.output)
output = Dense(10, activation="softmax")(x)

# Final model
model = Model(inputs=base_model.input, outputs=output)

model.summary()

17225924/17225924 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (None, 112, 112, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_bn (BatchNormalization)   │ (None, 112, 112, 32)   │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_relu (ReLU)               │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_1 (DepthwiseConv2D)     │ (None, 112, 112, 32)   │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_1_bn                    │ (None, 112, 112, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_1_relu (ReLU)           │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_1 (Conv2D)              │ (None, 112, 112, 64)   │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_1_bn                    │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_1_relu (ReLU)           │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pad_2 (ZeroPadding2D)      │ (None, 113, 113, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_2 (DepthwiseConv2D)     │ (None, 56, 56, 64)     │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_2_bn                    │ (None, 56, 56, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_2_relu (ReLU)           │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_2 (Conv2D)              │ (None, 56, 56, 128)    │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_2_bn                    │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_2_relu (ReLU)           │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_3 (DepthwiseConv2D)     │ (None, 56, 56, 128)    │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_3_bn                    │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_3_relu (ReLU)           │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_3 (Conv2D)              │ (None, 56, 56, 128)    │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_3_bn                    │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │             

 Total params: 3,239,114 (12.36 MB)

 Trainable params: 3,217,226 (12.27 MB)

 Non-trainable params: 21,888 (85.50 KB)

### Key Hyperparameters

#### Width Multiplier (α)

MobileNet introduces $\alpha$ which reduces channels.

* α = 1.0 → full network
* α = 0.5 → half channels
* α = 0.25 → very tiny network

This trades accuracy for speed.

#### Resolution Multiplier

Input image size can also be reduced.

Examples:

* 224×224
* 192×192
* 160×160
* 128×128

Smaller input → faster inference.

## Versions of MobileNet

1. **MobileNetV1:** Introduced depthwise separable convolution
2. **MobileNetV2**
    * Introduced:
      * **Inverted residual blocks**
      * **Linear bottlenecks**
    * Better performance & efficiency
3. **MobileNetV3(Small and Large)**
    * Uses:
      * Neural architecture search (NAS)
      * Squeeze-and-Excitation (SE)
      * Hard-swish activation
    * Optimized for real-world deployment

[Read More](https://www.geeksforgeeks.org/computer-vision/what-is-mobilenet-v2/)

### Inverted Residual Concept

Traditional residual block:

```text
Wide → Narrow → Wide
```

MobileNetV2:

```text
Narrow → Wide → Narrow
```

This preserves efficiency.

### Linear Bottleneck

ReLU can destroy information in low-dimensional spaces.

So MobileNetV2 uses:

* ReLU in expanded layers
* Linear activation in bottleneck layers

Result:

* Better accuracy
* Better feature preservation

## Cat vs Dog Classification

### Input Image

Input:

$$
224 × 224 × 3
$$
RGB image.

### First Convolution

Extract low-level features:

* edges
* corners
* textures

Output might become:

$$
112 × 112 × 32
$$

### Depthwise Convolution

Each channel processed independently.

Example:

$$
32 \text{ } channels
$$
$$
\downarrow
$$
$$
32 \text{ } separate \text{ } 3×3 \text{ } filters
$$

This detects spatial patterns efficiently.

### Pointwise Convolution

Apply:
$$
1 × 1 
$$
convolutions to combine channel information.

Maybe:

$$
32 \text{ } channels \text{ } → \text{ } 64 \text{ } channels
$$

Now network learns more complex features.

### Repeat Blocks

As layers deepen:

Network learns:

* fur textures
* ears
* eyes
* shapes

### Global Average Pooling

* Instead of huge dense layers Take average of each feature map.
* This reduces parameters heavily.

### Softmax

Final probabilities:

```text
Cat: 0.92
Dog: 0.08
```

Prediction: Cat

## Charecterstics

### Why MobileNet is Efficient

| Model       | Parameters | Speed  | Mobile Friendly |
| ----------- | ---------- | ------ | --------------- |
| VGG16       | ~138M      | Slow   | No              |
| ResNet50    | ~25M       | Medium | Limited         |
| MobileNetV1 | ~4.2M      | Fast   | Yes             |
| MobileNetV2 | ~3.4M      | Faster | Excellent       |

### Transfer Learning with MobileNet

One huge advantage:

* You can use pretrained MobileNet models.

Instead of training from scratch:

1. Load pretrained weights
2. Replace final classifier
3. Fine-tune on your dataset

This works great for:

* Small datasets
* Faster training
* Edge AI

### Advantages of MobileNet

* Efficient
    * Low computation
    * Low memory usage
* Fast
    * Real-time inference
* Small Size
    * Good for mobile apps
* Flexible
    * Adjustable width and resolution

### Limitations

* Lower Accuracy  
    Compared to large models like:
      * EfficientNet
      * Vision Transformers
      * Large ResNets
* Less Powerful for Complex Tasks
    * Very difficult tasks may require larger architectures.

### MobileNet vs EfficientNet

| Feature          | MobileNet | EfficientNet          |
| ---------------- | --------- | --------------------- |
| Goal             | Speed     | Accuracy + efficiency |
| Mobile optimized | Excellent | Good                  |
| Complexity       | Lower     | Higher                |
| Edge deployment  | Excellent | Good                  |
| Accuracy         | Moderate  | Higher                |

# Single Shot Detector (SSD)

A **Single Shot Detector (SSD)** is a deep learning object detection algorithm that detects and classifies objects in an image in **one forward pass** of a neural network.

Unlike older methods such as:

* **R-CNN / Fast R-CNN / Faster R-CNN** → which first generate region proposals and then classify them,
* SSD performs:

  1. **Object localization** (finding bounding boxes)
  2. **Object classification** (predicting object class)

simultaneously in a single network evaluation. That is why it is called **Single Shot** Detector.

## SSD Architecture

A typical SSD model contains:

* **Base Network (Backbone)**
  Usually:
  * VGG16
  * ResNet
  * MobileNet

This extracts image features.

* **Extra Convolution Layers**
  Added after the backbone to detect objects at multiple scales.
* **Prediction Layers**
  These predict:
  * class probabilities
  * bounding box offsets

### Architecture Flow

1. Input Image
2. Backbone CNN (VGG/ResNet)
3. Feature Maps at Different Scales
4. Convolution Predictors
5. Class Scores + Bounding Boxes
6. Non-Maximum Suppression
7. Final Detections

### Multi-Scale Feature Maps

Objects can appear:

* small
* medium
* large

SSD detects objects from **multiple feature maps**.

| Feature Map Size | Detects            |
| ---------------- | ------------------ |
| 38×38            | Small objects      |
| 19×19            | Medium objects     |
| 10×10            | Larger objects     |
| 5×5              | Very large objects |

This is one of SSD’s biggest strengths.

### Default Boxes (Anchor Boxes)

SSD uses predefined boxes called:

* Anchor boxes
* Prior boxes
* Default boxes

At every feature map location, SSD places multiple boxes with:

* different scales
* different aspect ratios

Example aspect ratios:

* 1:1
* 2:1
* 1:2
* 3:1

Suppose feature map size is:

$$
4 × 4
$$

Each cell predicts:

* 4 default boxes

Total default boxes:

$$
4\times4\times4=64
$$

Each box predicts:

* class probabilities
* box coordinates

### Bounding Box Prediction

For each default box SSD predicts:

$$
(cx, cy, w, h)
$$

Where:

* `cx` = center x
* `cy` = center y
* `w` = width
* `h` = height

These are offsets relative to the default box.

### Classification Prediction

SSD also predicts:

```text
P(class | object)
```

| Object | Probability |
| ------ | ----------- |
| Dog    | 0.92        |
| Cat    | 0.03        |
| Car    | 0.01        |

### Training SSD

Training uses two losses:

1. Localization Loss
    * Measures how close predicted boxes are to ground truth.  
    * Usually Smooth L1 Loss
2. Confidence Loss
    * Measures classification accuracy.
    * Usually:
        * Softmax Loss
        * Cross Entropy

3. Total Loss
$$
L(x,c,l,g)=\frac{1}{N}\left(L_{conf}(x,c)+\alpha L_{loc}(x,l,g)\right)
$$
Where:

* $L_{conf}$ = classification loss
* $L_{loc}$ = localization loss
* $N$ = matched default boxes
* $\alpha$ = balancing parameter

### Matching Strategy

During training:

* Ground truth boxes are matched with default boxes having highest IoU
* IoU = Intersection over Union.

**IoU Formula**

$$
IoU=\frac{Area\ of\ Overlap}{Area\ of\ Union}
$$
If:

```text
IoU > 0.5
```

the anchor is considered positive.

### Non-Maximum Suppression (NMS)

Multiple boxes may detect the same object.

NMS:

1. Keeps highest confidence box
2. Removes overlapping lower-confidence boxes

This prevents duplicate detections.

In [6]:
from tensorflow.keras.applications import MobileNetV2


def build_backbone(input_shape=(300,300,3)):

    backbone = MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights="imagenet"
    )

    feature1 = backbone.get_layer(
        "block_6_expand_relu"
    ).output

    feature2 = backbone.get_layer(
        "out_relu"
    ).output

    return backbone.input, [feature1, feature2]

In [7]:
def extra_layers(x):

    features = []

    x = layers.Conv2D(
        256,
        1,
        activation="relu"
    )(x)

    x = layers.Conv2D(
        512,
        3,
        strides=2,
        padding="same",
        activation="relu"
    )(x)

    features.append(x)

    x = layers.Conv2D(
        128,
        1,
        activation="relu"
    )(x)

    x = layers.Conv2D(
        256,
        3,
        strides=2,
        padding="same",
        activation="relu"
    )(x)

    features.append(x)

    return features

In [8]:
import tensorflow as tf
from tensorflow.keras import layers


def ssd_head(feature_map,
             num_anchors,
             num_classes):

    cls = layers.Conv2D(
        num_anchors * num_classes,
        kernel_size=3,
        padding="same"
    )(feature_map)

    box = layers.Conv2D(
        num_anchors * 4,
        kernel_size=3,
        padding="same"
    )(feature_map)

    return cls, box

In [9]:
def SSD300(
    input_shape=(300,300,3),
    num_classes=21
):

    inputs, backbone_features = build_backbone(
        input_shape
    )

    feature_maps = list(backbone_features)

    feature_maps += extra_layers(
        backbone_features[-1]
    )

    cls_outputs = []
    box_outputs = []

    anchors_per_location = 6

    for fmap in feature_maps:

        cls, box = ssd_head(
            fmap,
            anchors_per_location,
            num_classes
        )

        cls = layers.Reshape(
            (-1, num_classes)
        )(cls)

        box = layers.Reshape(
            (-1, 4)
        )(box)

        cls_outputs.append(cls)
        box_outputs.append(box)

    cls_outputs = layers.Concatenate(
        axis=1
    )(cls_outputs)

    box_outputs = layers.Concatenate(
        axis=1
    )(box_outputs)

    return tf.keras.Model(
        inputs,
        [cls_outputs, box_outputs],
        name="SSD300"
    )

In [10]:
model = SSD300(
    input_shape=(300,300,3),
    num_classes=21
)

model.summary()

/tmp/ipykernel_16/2478766183.py:6: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  backbone = MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "SSD300"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 300, 300,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 150, 150,  │        864 │ input_layer_3[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 150, 150,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 150, 150,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 150, 150,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 150, 150,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 150, 150,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 150, 150,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 150, 150,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 150, 150,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 150, 150,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 150, 150,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 151, 151,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 75, 75,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 75, 75,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 75, 75,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 75, 75,    │      2,304 │ block_1_depthwis

 Total params: 7,151,512 (27.28 MB)

 Trainable params: 7,117,400 (27.15 MB)

 Non-trainable params: 34,112 (133.25 KB)

In [11]:
!wget http://download.tensorflow.org/models/object_detection/tf2/20200711/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8.tar.gz

!tar -xvf ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8.tar.gz

--2026-06-01 15:00:13--  http://download.tensorflow.org/models/object_detection/tf2/20200711/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8.tar.gz
Resolving download.tensorflow.org (download.tensorflow.org)... 172.253.155.207, 173.194.195.207, 209.85.145.207, ...
Connecting to download.tensorflow.org (download.tensorflow.org)|172.253.155.207|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 20515344 (20M) [application/x-tar]
Saving to: ‘ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8.tar.gz’

ssd_mobilenet_v2_fp 100%[===================>]  19.56M  --.-KB/s    in 0.1s    

2026-06-01 15:00:14 (156 MB/s) - ‘ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8.tar.gz’ saved [20515344/20515344]

ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/
ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/
ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/ckpt-0.data-00000-of-00001
ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/checkpoint
ssd_mobilenet_v2_fpnlite_320x

In [12]:
# Pre-trained SSD with MobileNet backbone
import tensorflow as tf

model = tf.saved_model.load(
    "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/saved_model"
)

model.signatures.keys()

KeysView(_SignatureMap({'serving_default': <ConcreteFunction (*, input_tensor: TensorSpec(shape=(1, None, None, 3), dtype=tf.uint8, name='input_tensor')) -> Dict[['raw_detection_boxes', TensorSpec(shape=(1, 12804, 4), dtype=tf.float32, name='raw_detection_boxes')], ['detection_multiclass_scores', TensorSpec(shape=(1, 100, 91), dtype=tf.float32, name='detection_multiclass_scores')], ['detection_classes', TensorSpec(shape=(1, 100), dtype=tf.float32, name='detection_classes')], ['detection_boxes', TensorSpec(shape=(1, 100, 4), dtype=tf.float32, name='detection_boxes')], ['raw_detection_scores', TensorSpec(shape=(1, 12804, 91), dtype=tf.float32, name='raw_detection_scores')], ['num_detections', TensorSpec(shape=(1,), dtype=tf.float32, name='num_detections')], ['detection_anchor_indices', TensorSpec(shape=(1, 100), dtype=tf.float32, name='detection_anchor_indices')], ['detection_scores', TensorSpec(shape=(1, 100), dtype=tf.float32, name='detection_scores')]] at 0x7CF7F4117AA0>}))

## Car and Dog Classifier

Suppose input image contains:

* 1 dog
* 2 cars

1. Input Image

$$
300 × 300
$$

SSD300 is a common version.

2. Feature Extraction: Backbone CNN extracts features.

Output feature maps:

* 38×38
* 19×19
* 10×10
* etc.

3. Default Boxes Generated

Suppose:

* 8732 default boxes total

Each predicts:

* 4 box offsets
* class scores

4. Predictions

Example predictions:

| Box | Class | Confidence |
| --- | ----- | ---------- |
| B1  | Dog   | 0.95       |
| B2  | Car   | 0.91       |
| B3  | Car   | 0.87       |
| B4  | Car   | 0.40       |

5. NMS Applied

B2 and B4 overlap heavily.

NMS removes B4.

Final output:

* Dog detected
* Two cars detected

## SSD Variants

Common SSD models:

| Model  | Input Size |
| ------ | ---------- |
| SSD300 | 300×300    |
| SSD512 | 512×512    |

SSD512:

* better accuracy
* slower speed

## Charecterstics

### Advantages

1. Fast
    * Real-time detection possible.
    * Example: 40–60 FPS on GPU.
2. Simple Architecture
    * No separate region proposal stage.
3. Multi-Scale Detection
    * Good for varying object sizes.
4. End-to-End Training
    * Entire model trained together.

### Disadvantages

1. Weak Small Object Detection  
    Compared to:
    * Faster R-CNN
    * YOLOv8
    * RetinaNet

2. Many Negative Anchors
    * Most anchor boxes contain no objects.This creates class imbalance.  
    * SSD uses hard negative mining to solve this.

### SSD vs YOLO vs Faster R-CNN

| Feature       | SSD      | YOLO      | Faster R-CNN |
| ------------- | -------- | --------- | ------------ |
| Speed         | Fast     | Very Fast | Slower       |
| Accuracy      | Good     | Good      | Excellent    |
| Two-stage?    | No       | No        | Yes          |
| Small objects | Moderate | Moderate  | Better       |
| Real-time     | Yes      | Yes       | Usually no   |